In [1]:
import pandas as pd, numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()
books = pd.read_csv("../data/books_with_emotions.csv")
embeddings = OpenAIEmbeddings()
db = Chroma(persist_directory="../data/chroma_db", embedding_function=embeddings)

In [7]:
# pick 3 random books
selected = books.sample(3)
picks = selected["isbn13"].tolist()
selected[["title", "authors"]]

,title,authors
4111,Writing with Intent,Margaret Atwood
301,Soul Mates,Thomas Moore
826,Pride and Prejudice,Jane Austen


In [8]:
descriptions = selected["description"].tolist()
vectors = embeddings.embed_documents(descriptions)
# create a new vector that averages the 3 books
avg = np.mean(vectors, axis=0).tolist()

In [14]:
# categories of the 3 entered books -- recommendations must be in this set
allowed_categories = set(selected["simple_categories"])
print("Allowed categories:", allowed_categories)

recs = db.similarity_search_by_vector(avg, k=50)   # pull more, filtering will thin it out
isbns = [int(r.page_content.strip('"').split()[0]) for r in recs]

result = books[books["isbn13"].isin(isbns)]
result = result[~result["isbn13"].isin(picks)]                          # drop the books you picked
result = result[result["simple_categories"].isin(allowed_categories)]   # keep only matching categories
result[["title", "authors", "simple_categories"]].head(10)

Allowed categories: {'Nonfiction', 'Fiction'}


,title,authors,simple_categories
298,The Complete Stories,Zora Neale Hurston,Fiction
300,The Infinite Plan,Isabel Allende,Fiction
404,Women,Charles Bukowski,Fiction
418,A Circle of Quiet,Madeleine L'Engle,Nonfiction
608,Collected Short Stories,Graham Greene,Fiction
637,Existentialists and Mystics,Iris Murdoch,Nonfiction
640,Wide Sargasso Sea,Jean Rhys,Nonfiction
786,A Room of One's Own,Virginia Woolf,Fiction
810,Seven Gothic Tales,Isak Dinesen,Fiction
828,The Turn of the Screw and The Aspern Papers,Henry James,Fiction


In [16]:
def recommend_from_books(picks, k=50, n=10):
    """Recommend books based on a list of picked isbn13s.
    Averages the picks' embeddings, searches, drops the picks, and keeps
    only books whose simple_categories matches one of the picks'."""
    
    selected = books[books["isbn13"].isin(picks)]
    allowed_categories = set(selected["simple_categories"])

    avg = np.mean(embeddings.embed_documents(selected["description"].tolist()), axis=0).tolist()

    recs = db.similarity_search_by_vector(avg, k=k)   # pull more, filtering thins it out
    isbns = [int(r.page_content.strip('"').split()[0]) for r in recs]

    result = books[books["isbn13"].isin(isbns)]
    result = result[~result["isbn13"].isin(picks)]                          # drop the picks
    result = result[result["simple_categories"].isin(allowed_categories)]   # keep matching categories
    return result[["title", "authors", "simple_categories"]].head(n).reset_index(drop=True)


recommend_from_books(picks)

,title,authors,simple_categories
0,The Complete Stories,Zora Neale Hurston,Fiction
1,The Infinite Plan,Isabel Allende,Fiction
2,Women,Charles Bukowski,Fiction
3,A Circle of Quiet,Madeleine L'Engle,Nonfiction
4,Collected Short Stories,Graham Greene,Fiction
5,Existentialists and Mystics,Iris Murdoch,Nonfiction
6,Wide Sargasso Sea,Jean Rhys,Nonfiction
7,A Room of One's Own,Virginia Woolf,Fiction
8,Seven Gothic Tales,Isak Dinesen,Fiction
9,The Turn of the Screw and The Aspern Papers,Henry James,Fiction


In [17]:
# Category-coherence test: pick 3 children's books, recommendations should also be children's.
kids = books[books["simple_categories"].str.startswith("Children's")]
kids_trio = kids.sample(3, random_state=7)
kids_isbns = kids_trio["isbn13"].tolist()

print("Picked (children's):")
for t, c in zip(kids_trio["title"], kids_trio["simple_categories"]):
    print(f"  - {t}  [{c}]")

rec = recommend_from_books(kids_isbns)

share = rec["simple_categories"].str.startswith("Children's").mean()
print(f"\nChildren's share of top {len(rec)} recommendations: {share:.0%}")
rec

Picked (children's):
  - The Littles and Their Amazing New Friend  [Children's Fiction]
  - Harry Potter and the Half-Blood Prince (Book 6)  [Children's Fiction]
  - The Classic Treasury of Hans Christian Andersen  [Children's Fiction]

Children's share of top 10 recommendations: 100%


,title,authors,simple_categories
0,Poppy's Return,Avi,Children's Fiction
1,The Wee Free Men,Terry Pratchett,Children's Fiction
2,"The Lion, the Witch and the Wardrobe Read-Alou...",C. S. Lewis;Pauline Baynes,Children's Fiction
3,Howl's Moving Castle,Diana Wynne Jones,Children's Fiction
4,The Enchanted Castle,E. Nesbit,Children's Fiction
5,Westmark,Lloyd Alexander,Children's Fiction
6,The Circus of Adventure,Enid Blyton,Children's Fiction
7,Young Warriors,Tamora Pierce;Josepha Sherman,Children's Fiction
8,The Journey,Kathryn Lasky,Children's Fiction
9,Harry Potter and the Sorcerer's Stone (Book 1),"Rowling, J.K.",Children's Fiction
